### Phần 1: Chọn lọc Features

In [108]:
import pandas as pd
import numpy as np
df = pd.read_csv('../data/CSDL_DSS1_20240109_1.csv')

# 2. Loại bỏ dòng index 0 (chứa đơn vị như degree, mg/L, ‰...)
df = df.drop(index=0).reset_index(drop=True)
cols_to_keep = [
    'Matram', 'longitude', 'latitude', 'Z.Elev', 'Date', 
    'Salinity', 'CL',
]
df_clean = df[cols_to_keep].copy()
df_clean.head()

,Matram,longitude,latitude,Z.Elev,Date,Salinity,CL
0,QLPH012016,105.68,9.3,0.66,29/2/2016,3.6,1949
1,QLPH012016,105.68,9.3,0.66,15/3/2016,4.6,271.29
2,QLPH012016,105.68,9.3,0.66,2/4/2016,5.4,3261.4
3,QLPH012016,105.68,9.3,0.66,15/4/2016,5.1,3819
4,QLPH012016,105.68,9.3,0.66,2/5/2016,5.6,3574


### Phần 2: Ép kiểu dữ liệu & Xử lý thời gian

In [109]:
# 1. Định nghĩa các cột dạng số
numeric_cols = ['longitude', 'latitude', 'Z.Elev', 'Salinity', 'CL']

# Ép kiểu dữ liệu sang float, nếu có ký tự lạ (như '-', khoảng trắng) sẽ bị ép thành NaN (Not a Number)
for col in numeric_cols:
    df_clean[col] = pd.to_numeric(df_clean[col], errors='coerce')

# 2. Chuyển đổi cột Date sang định dạng Datetime
# format đang là Ngày/Tháng/Năm (vd: 29/2/2016)
df_clean['Date'] = pd.to_datetime(df_clean['Date'], format='%d/%m/%Y', errors='coerce')

# 3. Loại bỏ các dòng rác không chứa thông tin ngày tháng hoặc mã trạm
df_clean = df_clean.dropna(subset=['Date', 'Matram'])
df_clean.head(10)

,Matram,longitude,latitude,Z.Elev,Date,Salinity,CL
0,QLPH012016,105.68,9.30,0.66,2016-02-29,3.6,1949.00
1,QLPH012016,105.68,9.30,0.66,2016-03-15,4.6,271.29
2,QLPH012016,105.68,9.30,0.66,2016-04-02,5.4,3261.40
3,QLPH012016,105.68,9.30,0.66,2016-04-15,5.1,3819.00
4,QLPH012016,105.68,9.30,0.66,2016-05-02,5.6,3574.00
5,QLPH012016,105.68,9.30,0.66,2016-05-16,3.0,1391.20
6,QLPH012016,105.68,9.30,0.66,2016-06-02,2.2,1195.60
7,QLPH012016,105.68,9.30,0.66,2016-06-15,1.2,795.40
8,QLPH022016,105.53,9.26,0.54,2016-02-29,1.6,918.00
9,QLPH022016,105.53,9.26,0.54,2016-03-15,1.7,16.78


### Phần 3: Tiền xử lý bất thường Độ mặn & Clorua

In [110]:
# 1. Đưa các giá trị 0 ngụy trang về dạng NaN để tránh nhiễu model
# Trực quan: Nếu mặn > 0 mà Clo = 0 -> Clo bị đo lỗi
df_clean.loc[(df_clean['Salinity'] > 0) & (df_clean['CL'] == 0), 'CL'] = np.nan

# Trực quan: Nếu Clo > 0 mà mặn = 0 -> Mặn bị đo lỗi (dưới ngưỡng phát hiện của máy đo)
df_clean.loc[(df_clean['CL'] > 0) & (df_clean['Salinity'] == 0), 'Salinity'] = np.nan
df_clean

,Matram,longitude,latitude,Z.Elev,Date,Salinity,CL
0,QLPH012016,105.68,9.30,0.66,2016-02-29,3.60,1949.00
1,QLPH012016,105.68,9.30,0.66,2016-03-15,4.60,271.29
2,QLPH012016,105.68,9.30,0.66,2016-04-02,5.40,3261.40
3,QLPH012016,105.68,9.30,0.66,2016-04-15,5.10,3819.00
4,QLPH012016,105.68,9.30,0.66,2016-05-02,5.60,3574.00
...,...,...,...,...,...,...,...
2691,BD00142021,106.45,10.35,0.33,2021-01-29,3.10,673.50
2692,BD00152021,106.33,10.34,0.35,2021-01-29,0.35,53.49
2693,BD00162021,106.25,10.42,0.35,2021-01-29,0.16,22.65
2694,BD00172021,106.33,10.48,0.35,2021-01-29,0.12,16.50


### Phần 3.5: Lấp khuyết (Impute) dữ liệu NaN cho Salinity và CL

In [111]:
print(f"Số lượng NaN trước xử lý: Salinity ({df_clean['Salinity'].isna().sum()}), CL ({df_clean['CL'].isna().sum()})")

# ==========================================
# CHIẾN LƯỢC 1: NỘI SUY CHÉO VẬT LÝ
# Dựa trên hệ thức Knudsen của Hải dương học: Độ mặn (‰) ≈  0.00180665 × Clorua (mg/l)
# ==========================================

# 1.1 Điền Salinity bị NaN dựa vào giá trị CL hiện có
cond_sal_nan = df_clean['Salinity'].isna() & df_clean['CL'].notna()
df_clean.loc[cond_sal_nan, 'Salinity'] = df_clean.loc[cond_sal_nan, 'CL'] *  0.00180665

# 1.2 Điền CL bị NaN dựa vào giá trị Salinity hiện có
cond_cl_nan = df_clean['CL'].isna() & df_clean['Salinity'].notna()
df_clean.loc[cond_cl_nan, 'CL'] = (df_clean.loc[cond_cl_nan, 'Salinity'] /  0.00180665)


# ==========================================
# CHIẾN LƯỢC 2: NỘI SUY CHUỖI THỜI GIAN THEO TRẠM
# Dành cho các dòng bị NaN cả Salinity lẫn CL
# ==========================================

# sort dữ liệu theo Trạm và Thời gian thì hàm Interpolate mới chạy đúng logic vật lý
df_clean = df_clean.sort_values(by=['Matram', 'Date']).reset_index(drop=True)

# Dùng groupby để gom theo từng trạm, sau đó nội suy tuyến tính (linear) 
# limit=3: Chỉ nội suy tối đa 3 ngày liên tiếp bị thiếu để tránh bịa data quá đà
df_clean[['Salinity', 'CL']] = df_clean.groupby('Matram')[['Salinity', 'CL']].apply(
    lambda group: group.interpolate(method='linear', limit=3, limit_direction='both')
).reset_index(drop=True)

# Lỡ còn sót lại NaN nào do ở đầu/cuối chuỗi dữ liệu của trạm đó, fill bằng 0 hoặc trung bình trạm
# Ở đây ta lấp nốt bằng backward/forward fill trong chính trạm đó
df_clean[['Salinity', 'CL']] = df_clean.groupby('Matram')[['Salinity', 'CL']].apply(
    lambda group: group.bfill().ffill()
).reset_index(drop=True)

print(f"Số lượng NaN sau xử lý: Salinity ({df_clean['Salinity'].isna().sum()}), CL ({df_clean['CL'].isna().sum()})")

Số lượng NaN trước xử lý: Salinity (665), CL (323)
Số lượng NaN sau xử lý: Salinity (0), CL (0)


### Phần 4: Tiền xử lý dữ liệu mực nước thượng nguồn (Tân Châu)

In [112]:
df_tanchau = pd.read_json('../data/tanChauwaterLevel2016.json') 
# 1. Ép kiểu thời gian cho cột ObservationDate
df_tanchau['ObservationDate'] = pd.to_datetime(df_tanchau['ObservationDate'], errors='coerce')

# 2. Gom nhóm theo ngày (Daily Resampling) để tính Mean, Max, Min
df_tc_daily = df_tanchau.groupby(df_tanchau['ObservationDate'].dt.date).agg(
    WaterLevel_Mean=('WaterLevel', 'mean'),
    WaterLevel_Max=('WaterLevel', 'max'),
    WaterLevel_Min=('WaterLevel', 'min')
).reset_index()

# Đưa cột ObservationDate về lại chuẩn Datetime để Merge
df_tc_daily['ObservationDate'] = pd.to_datetime(df_tc_daily['ObservationDate'])

# 3.Tạo Time-Lag Features cho thượng nguồn
# Tính trung bình mực nước của 3 ngày và 7 ngày trước đó
df_tc_daily = df_tc_daily.sort_values('ObservationDate').set_index('ObservationDate')
df_tc_daily['WL_Mean_3D_Lag'] = df_tc_daily['WaterLevel_Mean'].shift(1).rolling(window=3).mean()
df_tc_daily['WL_Mean_7D_Lag'] = df_tc_daily['WaterLevel_Mean'].shift(1).rolling(window=7).mean()
df_tc_daily = df_tc_daily.reset_index()

print("Hoàn thành Phần 4: Kích thước dữ liệu Tân Châu theo ngày:", df_tc_daily.shape)

Hoàn thành Phần 4: Kích thước dữ liệu Tân Châu theo ngày: (3646, 6)


### Phần 5: Tính khoảng cách Không gian (Spatial Distance)
##### Sử dụng công thức Haversine để tính khoảng cách đường chim bay từ tọa độ Tân Châu đến tọa độ của từng dòng quan trắc trong df_clean

In [ ]:
# Hàm tính khoảng cách Haversine
def haversine_distance(lat1, lon1, lat2, lon2):
    R = 6371.0 # Bán kính Trái Đất (km)
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2)**2
    c = 2 * np.arcsin(np.sqrt(a))
    return R * c

# Tọa độ trạm Tân Châu 
TAN_CHAU_LAT = 10.778
TAN_CHAU_LON = 105.205 

# Tính khoảng cách đường chim bay (km)
df_clean['Dist_to_TanChau_km'] = haversine_distance(
    TAN_CHAU_LAT, TAN_CHAU_LON, 
    df_clean['latitude'], df_clean['longitude']
)

# Nhân với hệ số uốn khúc của sông (Sông Mê Kông thường xấp xỉ 1.3 - 1.4)
SINUOSITY_INDEX = 1.3
df_clean['River_Dist_km'] = df_clean['Dist_to_TanChau_km'] * SINUOSITY_INDEX
print("Hoàn thành Phần 5: Đã tính xong khoảng cách địa lý cho các trạm.")

Hoàn thành Phần 5: Đã tính xong khoảng cách địa lý cho các trạm.
Khoảng cách xa nhất từ trạm Tân Châu đến trạm đo chất lượng nước: 178.84922610840937


In [123]:
maxDist_to_TanChau_km = df_clean['Dist_to_TanChau_km'].max()
print("Khoảng cách xa nhất từ trạm Tân Châu đến trạm đo chất lượng nước: " + (str)(maxDist_to_TanChau_km))

Khoảng cách xa nhất từ trạm Tân Châu đến trạm đo chất lượng nước: 178.84922610840937


#### Tọa độ trạm xa trạm Tân Châu nhất

In [ ]:
df_clean.loc[(df_clean['Dist_to_TanChau_km'] == maxDist_to_TanChau_km),['longitude', 'latitude']]

,longitude,latitude
1183,106.38,9.66
1184,106.38,9.66
1185,106.38,9.66
1186,106.38,9.66
1187,106.38,9.66
1188,106.38,9.66
1189,106.38,9.66
1190,106.38,9.66
1191,106.38,9.66
1192,106.38,9.66


### Phần 6: Tính Thời gian truyền lũ (Time Lag Calculation)
##### Dựa vào vận tốc dòng chảy trung bình để quy đổi khoảng cách (km) thành thời gian chảy (ngày).

In [ ]:
# Vận tốc dòng chảy trung bình (km/ngày). 
# Bạn có thể tuning con số này sau khi chạy thử model (Dao động 40 - 60 km/ngày)
AVERAGE_FLOW_SPEED_KMDAY = 50.0 

# Tính số ngày chảy và làm tròn thành số nguyên (integer)
df_clean['Travel_Time_Days'] = np.round(df_clean['River_Dist_km'] / AVERAGE_FLOW_SPEED_KMDAY).astype(int)

# Đảm bảo cột Date của df_clean đang ở định dạng Datetime
df_clean['Date'] = pd.to_datetime(df_clean['Date'])

# Lùi ngày đo đạc lại đúng bằng số ngày trễ để tạo cột "Ngày mục tiêu"
df_clean['TanChau_Target_Date'] = df_clean['Date'] - pd.to_timedelta(df_clean['Travel_Time_Days'], unit='D')

print("Hoàn thành Phần 6: Mẫu dữ liệu sau khi lùi ngày:")
print(df_clean[['Matram', 'Date', 'Travel_Time_Days', 'TanChau_Target_Date']].head())

Hoàn thành Phần 6: Mẫu dữ liệu sau khi lùi ngày:
       Matram       Date  Travel_Time_Days TanChau_Target_Date
0  BD00012019 2019-03-04                 4          2019-02-28
1  BD00012019 2019-03-18                 4          2019-03-14
2  BD00012019 2019-04-02                 4          2019-03-29
3  BD00012019 2019-04-16                 4          2019-04-12
4  BD00012019 2019-05-01                 4          2019-04-27


### Phần 7: Ghép động (Dynamic Merge) và Dọn dẹp
##### Tiến hành gộp 2 bảng dữ liệu lại với nhau dựa trên cái "Ngày mục tiêu" vừa tính được.

In [ ]:
# 1. Thực hiện Left Join
# Lấy df_clean làm gốc, tìm mực nước Tân Châu tương ứng vào ngày 'TanChau_Target_Date'
df_final = pd.merge(
    df_clean,
    df_tc_daily,
    left_on='TanChau_Target_Date',
    right_on='ObservationDate',
    how='left'
)

# 2. Dọn dẹp (Drop) các cột tạm thời không còn giá trị sử dụng để làm nhẹ data
cols_to_drop = ['TanChau_Target_Date', 'ObservationDate']
df_final = df_final.drop(columns=cols_to_drop, errors='ignore')

# 3. Điền khuyết dữ liệu thượng nguồn nếu có ngày bị trống
water_cols = ['WaterLevel_Mean', 'WaterLevel_Max', 'WaterLevel_Min', 'WL_Mean_3D_Lag', 'WL_Mean_7D_Lag']

# Sử dụng interpolate tuyến tính. 
# limit=7: Đề phòng nếu mất dữ liệu quá dài (>7 ngày) thì không tự bịa ra quá nhiều để tránh rủi ro.
df_final[water_cols] = df_final[water_cols].interpolate(method='linear', limit=7, limit_direction='forward')
print(f"Kích thước DataFrame cuối cùng: {df_final.shape}")
df_final

Kích thước DataFrame cuối cùng: (2696, 15)


,Matram,longitude,latitude,Z.Elev,Date,Salinity,CL,Dist_to_TanChau_km,River_Dist_km,Travel_Time_Days,WaterLevel_Mean,WaterLevel_Max,WaterLevel_Min,WL_Mean_3D_Lag,WL_Mean_7D_Lag
0,BD00012019,106.58,10.43,0.95,2019-03-04,8.100000,746.80,155.183543,201.738606,4,0.68,0.68,0.68,0.300000,0.385714
1,BD00012019,106.58,10.43,0.95,2019-03-18,9.000000,921.00,155.183543,201.738606,4,0.26,0.26,0.26,0.100000,0.348571
2,BD00012019,106.58,10.43,0.95,2019-04-02,8.000000,91.56,155.183543,201.738606,4,0.57,0.57,0.57,0.380000,0.501429
3,BD00012019,106.58,10.43,0.95,2019-04-16,9.400000,1832.00,155.183543,201.738606,4,0.24,0.24,0.24,0.133333,0.542857
4,BD00012019,106.58,10.43,0.95,2019-05-01,6.700000,1526.00,155.183543,201.738606,4,0.37,0.37,0.37,0.260000,0.475714
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2691,VC00102021,106.26,11.08,0.72,2021-05-11,0.100000,19.90,119.978096,155.971525,3,1.09,1.09,1.09,0.296667,0.107143
2692,VC00102021,106.26,11.08,0.72,2021-05-26,0.035952,19.90,119.978096,155.971525,3,0.35,0.35,0.35,-0.090000,-0.201429
2693,VC00102021,106.26,11.08,0.72,2021-06-10,0.051309,28.40,119.978096,155.971525,3,0.78,0.78,0.78,0.196667,-0.032857
2694,VC00102021,106.26,11.08,0.72,2021-06-23,0.023125,12.80,119.978096,155.971525,3,0.43,0.43,0.43,0.146667,-0.082857


### Phần 8: Feature Engineering

In [ ]:

print("Kích thước dữ liệu trước Feature Engineering:", df_final.shape)

# ==========================================
# TRÍCH XUẤT ĐẶC TRƯNG THỜI GIAN TỪ 'Date'
# ==========================================
# Lấy ra Tháng và Ngày trong năm
df_final['Month'] = df_final['Date'].dt.month
df_final['DayOfYear'] = df_final['Date'].dt.dayofyear

# Mã hóa tuần hoàn - Time-Series mùa vụ
# Quy đổi chu kỳ 12 tháng
df_final['Month_sin'] = np.sin(2 * np.pi * df_final['Month'] / 12)
df_final['Month_cos'] = np.cos(2 * np.pi * df_final['Month'] / 12)

# Quy đổi chu kỳ 365 ngày
df_final['Day_sin'] = np.sin(2 * np.pi * df_final['DayOfYear'] / 365)
df_final['Day_cos'] = np.cos(2 * np.pi * df_final['DayOfYear'] / 365)


# Xóa các cột mô hình không hiểu
cols_to_drop_for_training = [ 'Month', 'DayOfYear', 'CL', 'Travel_Time_Days', 'WaterLevel_Mean', 'WaterLevel_Max', 'WaterLevel_Min', 'River_Dist_km']
df_train_ready = df_final.drop(columns=cols_to_drop_for_training)

print("Kích thước dữ liệu sau khi Drop cột rác:", df_train_ready.shape)

Kích thước dữ liệu trước Feature Engineering: (2696, 15)
Kích thước dữ liệu sau khi Drop cột rác: (2696, 13)


 **Bắt buộc phải dùng CẢ HAI (Sin và Cos). Nếu chỉ dùng 1, bạn sẽ vô tình "phá hỏng" dữ liệu của mình.**

giải thích lý do:

### Sự cố "Trùng lặp điểm mù" (Blind-spot Overlap)

Bản chất của hàm Sin và Cos là ánh xạ (map) một đường thẳng thời gian (từ tháng 1 đến tháng 12) cuộn lại thành **một vòng tròn 2D (giống như mặt đồng hồ)**.

Hãy thử tưởng tượng **chỉ dùng hàm SIN** cho chu kỳ 12 tháng: `Month_sin = sin(2 * pi * Month / 12)`.

* Trị số của **Tháng 6**: `sin(2 * pi * 6 / 12)` = `sin(pi)` = **0**
* Trị số của **Tháng 12**: `sin(2 * pi * 12 / 12)` = `sin(2*pi)` = **0**

**Chuyện gì sẽ xảy ra với mô hình AI của bạn?**
Thuật toán sẽ nhìn vào dữ liệu và thấy Tháng 6 và Tháng 12 đều có giá trị `Month_sin = 0`. Nó sẽ kết luận: *"À, Tháng 6 và Tháng 12 giống hệt nhau!"*.
Nhưng trên thực tế ở Đồng bằng sông Cửu Long, Tháng 6 là đầu mùa mưa (nước ngọt vô vàn), còn Tháng 12 là đầu mùa khô (nước mặn chát). Việc AI nhầm lẫn 2 tháng này với nhau là một **thảm họa dự báo**.

Hiện tượng trùng lặp này cũng xảy ra tương tự với các cặp tháng khác (ví dụ Tháng 5 và Tháng 7 cũng cho ra giá trị Sin giống hệt nhau).

### Sự "Cứu rỗi" của Cosine

Để khắc phục điểm mù này, chúng ta phải đưa thêm tọa độ **Cos** vào. Vẫn với ví dụ trên: `Month_cos = cos(2 * pi * Month / 12)`

* Tọa độ của **Tháng 6**: `Month_sin` = 0, `Month_cos` = **-1** => Điểm trên đồ thị: **(0, -1)**
* Tọa độ của **Tháng 12**: `Month_sin` = 0, `Month_cos` = **1** => Điểm trên đồ thị: **(0, 1)**

Lúc này, mô hình Machine Learning sẽ nhận được một **tọa độ 2 chiều (X, Y) = (Cos, Sin)**. Khi đối chiếu tọa độ (0, -1) và (0, 1), AI sẽ hiểu ra ngay lập tức: *"Hóa ra đây là 2 thời điểm nằm ở 2 cực đối diện nhau hoàn toàn trên vòng tròn thời gian (cách nhau đúng nửa năm)"*.

**Tóm lại:** Bạn đang cố gắng vẽ một vòng tròn. Để xác định duy nhất và chính xác một vị trí trên vòng tròn đó, bạn luôn cần 2 tọa độ (Trục dọc Y là Sin, Trục ngang X là Cos). Dùng 1 cái thì vòng tròn sẽ bị "bóp bẹp" thành 1 đường thẳng, và các tháng ở hai nửa năm sẽ đè lên nhau gây sai lệch mô hình!

In [ ]:
df_train_ready.to_csv('../data/processed/Final__Data_Salinity.csv', index=False)